# DS605: Fundamentals of Machine Learning
## Lab Assignment - 5
### Machine Learning with Scikit-learn and From Scratch

**Dataset:** UCI Productivity Prediction of Garment Employees

This notebook implements the assignment workflow:
**Raw Data → Preprocessing → Train-Test Split → Model Training → Prediction → Evaluation → Comparison → Optimization**

The manual section uses only **NumPy and Pandas** for machine-learning implementation, as required by the assignment.

## Assignment Requirements

- Regression: predict `actual_productivity` using Linear Regression.
- Classification: create `MeetsTarget = 1` when `actual_productivity >= targeted_productivity`, otherwise `0`.
- Do not use `actual_productivity` as a classification input.
- Use the same fixed train-test split for all comparisons.
- Scikit-learn section: preprocessing, Linear Regression, Logistic Regression and metrics.
- From-scratch section: missing values, categorical encoding, scaling, Linear Regression, Logistic Regression, predictions and metrics using NumPy/Pandas.
- Compare predictive performance, training time and prediction time.
- Optimize the manual implementation and explain differences.

In [2]:

# ============================================================
# 1. Imports and configuration
# ============================================================

import numpy as np
import pandas as pd
import time
from pathlib import Path

RANDOM_STATE = 42
TEST_SIZE = 0.20

# Change this if your CSV has a different filename/path.
DATA_PATH = r"C:\Users\Raaj Soni\OneDrive\Desktop\202618039_ML_assignment\lab_05\garments_worker_productivity.csv"

print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)


NumPy version: 2.5.1
Pandas version: 3.0.5


In [3]:

# ============================================================
# 2. Load dataset
# ============================================================

if not Path(DATA_PATH).exists():
    # Try common filenames automatically
    candidates = [
        "garments_worker_productivity.csv",
        "garment_worker_productivity.csv",
        "garment_productivity.csv",
        "train.csv"
    ]
    found = next((p for p in candidates if Path(p).exists()), None)
    if found is not None:
        DATA_PATH = found

if not Path(DATA_PATH).exists():
    raise FileNotFoundError(
        f"Dataset not found. Put the UCI garment productivity CSV in the "
        f"same folder as this notebook and set DATA_PATH. Current path: {DATA_PATH}"
    )

df = pd.read_csv(r"C:\Users\Raaj Soni\OneDrive\Desktop\202618039_ML_assignment\lab_05\garments_worker_productivity.csv")

print("Shape:", df.shape)
display(df.head())
display(df.info())


Shape: (1197, 15)


,date,quarter,department,day,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,actual_productivity
0,1/1/2015,Quarter1,sweing,Thursday,8,0.80,26.16,1108.0,7080,98,0.0,0,0,59.0,0.940725
1,1/1/2015,Quarter1,finishing,Thursday,1,0.75,3.94,NaN,960,0,0.0,0,0,8.0,0.886500
2,1/1/2015,Quarter1,sweing,Thursday,11,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
3,1/1/2015,Quarter1,sweing,Thursday,12,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
4,1/1/2015,Quarter1,sweing,Thursday,6,0.80,25.90,1170.0,1920,50,0.0,0,0,56.0,0.800382


<class 'pandas.DataFrame'>
RangeIndex: 1197 entries, 0 to 1196
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   date                   1197 non-null   str    
 1   quarter                1197 non-null   str    
 2   department             1197 non-null   str    
 3   day                    1197 non-null   str    
 4   team                   1197 non-null   int64  
 5   targeted_productivity  1197 non-null   float64
 6   smv                    1197 non-null   float64
 7   wip                    691 non-null    float64
 8   over_time              1197 non-null   int64  
 9   incentive              1197 non-null   int64  
 10  idle_time              1197 non-null   float64
 11  idle_men               1197 non-null   int64  
 12  no_of_style_change     1197 non-null   int64  
 13  no_of_workers          1197 non-null   float64
 14  actual_productivity    1197 non-null   float64
dtypes: float64(6), 

None

## 3. Initial Data Inspection

In [4]:

print("Columns:")
print(df.columns.tolist())

print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False))

print("\nData types:")
display(df.dtypes)

print("\nSummary:")
display(df.describe(include="all").T)


Columns:
['date', 'quarter', 'department', 'day', 'team', 'targeted_productivity', 'smv', 'wip', 'over_time', 'incentive', 'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers', 'actual_productivity']

Missing values:


wip                      506
quarter                    0
date                       0
day                        0
team                       0
targeted_productivity      0
department                 0
smv                        0
over_time                  0
incentive                  0
idle_time                  0
idle_men                   0
no_of_style_change         0
no_of_workers              0
actual_productivity        0
dtype: int64


Data types:


date                         str
quarter                      str
department                   str
day                          str
team                       int64
targeted_productivity    float64
smv                      float64
wip                      float64
over_time                  int64
incentive                  int64
idle_time                float64
idle_men                   int64
no_of_style_change         int64
no_of_workers            float64
actual_productivity      float64
dtype: object


Summary:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
date,1197,59,1/31/2015,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN
quarter,1197,5,Quarter1,360,NaN,NaN,NaN,NaN,NaN,NaN,NaN
department,1197,3,sweing,691,NaN,NaN,NaN,NaN,NaN,NaN,NaN
day,1197,6,Wednesday,208,NaN,NaN,NaN,NaN,NaN,NaN,NaN
team,1197.0,NaN,NaN,NaN,6.426901,3.463963,1.0,3.0,6.0,9.0,12.0
targeted_productivity,1197.0,NaN,NaN,NaN,0.729632,0.097891,0.07,0.7,0.75,0.8,0.8
smv,1197.0,NaN,NaN,NaN,15.062172,10.943219,2.9,3.94,15.26,24.26,54.56
wip,691.0,NaN,NaN,NaN,1190.465991,1837.455001,7.0,774.5,1039.0,1252.5,23122.0
over_time,1197.0,NaN,NaN,NaN,4567.460317,3348.823563,0.0,1440.0,3960.0,6960.0,25920.0
incentive,1197.0,NaN,NaN,NaN,38.210526,160.182643,0.0,0.0,0.0,50.0,3600.0


## 4. Clean Column Names

The UCI dataset sometimes contains whitespace in column names. The following step standardizes only the column-name formatting while preserving the dataset's variables.

In [5]:

df = df.copy()
df.columns = df.columns.str.strip()

print(df.columns.tolist())


['date', 'quarter', 'department', 'day', 'team', 'targeted_productivity', 'smv', 'wip', 'over_time', 'incentive', 'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers', 'actual_productivity']


In [6]:

# Verify required target columns
required = {"actual_productivity", "targeted_productivity"}
missing_required = required - set(df.columns)

if missing_required:
    raise ValueError(
        f"Required columns are missing: {missing_required}. "
        f"Available columns: {df.columns.tolist()}"
    )

# Create classification target exactly as required
df["MeetsTarget"] = (
    df["actual_productivity"] >= df["targeted_productivity"]
).astype(int)

print(df["MeetsTarget"].value_counts())
display(df[["targeted_productivity", "actual_productivity", "MeetsTarget"]].head())


MeetsTarget
1    875
0    322
Name: count, dtype: int64


,targeted_productivity,actual_productivity,MeetsTarget
0,0.80,0.940725,1
1,0.75,0.886500,1
2,0.80,0.800570,1
3,0.80,0.800570,1
4,0.80,0.800382,1


## 5. Define Features and Targets

For classification, `actual_productivity` must not be used as an input because it directly defines the target.

In [7]:

REGRESSION_TARGET = "actual_productivity"
CLASSIFICATION_TARGET = "MeetsTarget"

# Actual productivity is excluded from classification features.
classification_drop = {
    REGRESSION_TARGET,
    CLASSIFICATION_TARGET
}

# Targeted productivity is retained because it is a legitimate predictor.
X_reg_raw = df.drop(columns=[REGRESSION_TARGET, CLASSIFICATION_TARGET])
y_reg = df[REGRESSION_TARGET].astype(float)

X_clf_raw = df.drop(columns=list(classification_drop))
y_clf = df[CLASSIFICATION_TARGET].astype(int)

print("Regression features:", X_reg_raw.shape)
print("Classification features:", X_clf_raw.shape)
print("Regression target shape:", y_reg.shape)
print("Classification target shape:", y_clf.shape)


Regression features: (1197, 14)
Classification features: (1197, 14)
Regression target shape: (1197,)
Classification target shape: (1197,)


# Part A — Scikit-learn Implementation

A single fixed train-test split is created first. The same row split will then be reused by the manual implementation.

In [8]:

# ============================================================
# 6. Fixed train-test split
# ============================================================

from sklearn.model_selection import train_test_split

indices = np.arange(len(df))

train_idx, test_idx = train_test_split(
    indices,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True
)

print("Train samples:", len(train_idx))
print("Test samples :", len(test_idx))

# Same exact row split for both tasks
X_reg_train_raw = X_reg_raw.iloc[train_idx].copy()
X_reg_test_raw  = X_reg_raw.iloc[test_idx].copy()
y_reg_train = y_reg.iloc[train_idx].to_numpy()
y_reg_test  = y_reg.iloc[test_idx].to_numpy()

X_clf_train_raw = X_clf_raw.iloc[train_idx].copy()
X_clf_test_raw  = X_clf_raw.iloc[test_idx].copy()
y_clf_train = y_clf.iloc[train_idx].to_numpy()
y_clf_test  = y_clf.iloc[test_idx].to_numpy()


Train samples: 957
Test samples : 240


## 7. Scikit-learn Preprocessing

- Numeric features: median imputation + standard scaling.
- Categorical features: most-frequent imputation + one-hot encoding.
- The preprocessing objects are fitted only on training data.

In [9]:

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def make_preprocessor(X):
    numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
    categorical_cols = X.select_dtypes(exclude=np.number).columns.tolist()

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, numeric_cols),
        ("cat", categorical_pipeline, categorical_cols)
    ])

    return preprocessor

reg_preprocessor = make_preprocessor(X_reg_train_raw)
clf_preprocessor = make_preprocessor(X_clf_train_raw)

X_reg_train = reg_preprocessor.fit_transform(X_reg_train_raw)
X_reg_test = reg_preprocessor.transform(X_reg_test_raw)

X_clf_train = clf_preprocessor.fit_transform(X_clf_train_raw)
X_clf_test = clf_preprocessor.transform(X_clf_test_raw)

print("Regression transformed shape:", X_reg_train.shape)
print("Classification transformed shape:", X_clf_train.shape)


Regression transformed shape: (957, 83)
Classification transformed shape: (957, 83)


In [10]:

# ============================================================
# 8. Scikit-learn Linear Regression
# ============================================================

from sklearn.linear_model import LinearRegression

sk_reg_model = LinearRegression()

start = time.perf_counter()
sk_reg_model.fit(X_reg_train, y_reg_train)
sk_reg_train_time = time.perf_counter() - start

start = time.perf_counter()
sk_reg_pred = sk_reg_model.predict(X_reg_test)
sk_reg_pred_time = time.perf_counter() - start

print("Training time :", sk_reg_train_time)
print("Prediction time:", sk_reg_pred_time)


Training time : 0.012818499992135912
Prediction time: 0.0010320000001229346


In [11]:

# ============================================================
# 9. Regression metrics
# ============================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sk_reg_mae = mean_absolute_error(y_reg_test, sk_reg_pred)
sk_reg_rmse = np.sqrt(mean_squared_error(y_reg_test, sk_reg_pred))
sk_reg_r2 = r2_score(y_reg_test, sk_reg_pred)

print(f"MAE : {sk_reg_mae:.6f}")
print(f"RMSE: {sk_reg_rmse:.6f}")
print(f"R2  : {sk_reg_r2:.6f}")


MAE : 0.112096
RMSE: 0.150645
R2  : 0.145316


In [12]:

# ============================================================
# 10. Scikit-learn Logistic Regression
# ============================================================

from sklearn.linear_model import LogisticRegression

sk_clf_model = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)

start = time.perf_counter()
sk_clf_model.fit(X_clf_train, y_clf_train)
sk_clf_train_time = time.perf_counter() - start

start = time.perf_counter()
sk_clf_pred = sk_clf_model.predict(X_clf_test)
sk_clf_pred_time = time.perf_counter() - start

print("Training time :", sk_clf_train_time)
print("Prediction time:", sk_clf_pred_time)


Training time : 4.920923400000902
Prediction time: 0.0005606000049738213


In [13]:

# ============================================================
# 11. Classification metrics
# ============================================================

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

sk_clf_accuracy = accuracy_score(y_clf_test, sk_clf_pred)
sk_clf_precision = precision_score(y_clf_test, sk_clf_pred, zero_division=0)
sk_clf_recall = recall_score(y_clf_test, sk_clf_pred, zero_division=0)
sk_clf_f1 = f1_score(y_clf_test, sk_clf_pred, zero_division=0)

print(f"Accuracy : {sk_clf_accuracy:.6f}")
print(f"Precision: {sk_clf_precision:.6f}")
print(f"Recall   : {sk_clf_recall:.6f}")
print(f"F1-score : {sk_clf_f1:.6f}")


Accuracy : 0.750000
Precision: 0.774648
Recall   : 0.932203
F1-score : 0.846154


# Part B — From-Scratch Implementation

**Important:** This section intentionally does not use Scikit-learn preprocessing, models, metrics, or train-test utilities. NumPy and Pandas are used for the complete manual workflow.

In [14]:

# ============================================================
# 12. Manual preprocessing utilities
# ============================================================

def manual_fit_preprocessor(X_train):
    X_train = X_train.copy()

    numeric_cols = X_train.select_dtypes(include=np.number).columns.tolist()
    categorical_cols = X_train.select_dtypes(exclude=np.number).columns.tolist()

    numeric_medians = {}
    numeric_means = {}
    numeric_stds = {}

    for col in numeric_cols:
        values = pd.to_numeric(X_train[col], errors="coerce")
        median = values.median()
        values = values.fillna(median)

        mean = values.mean()
        std = values.std(ddof=0)

        if not np.isfinite(std) or std == 0:
            std = 1.0

        numeric_medians[col] = median
        numeric_means[col] = mean
        numeric_stds[col] = std

    category_values = {}
    category_modes = {}

    for col in categorical_cols:
        s = X_train[col].astype("object")
        mode = s.mode(dropna=True)
        fill_value = mode.iloc[0] if len(mode) else "Missing"

        s = s.fillna(fill_value).astype(str)
        categories = sorted(s.unique().tolist())

        category_modes[col] = str(fill_value)
        category_values[col] = categories

    return {
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
        "numeric_medians": numeric_medians,
        "numeric_means": numeric_means,
        "numeric_stds": numeric_stds,
        "category_values": category_values,
        "category_modes": category_modes
    }


def manual_transform(X, prep):
    X = X.copy()
    blocks = []

    # Numeric: impute + standardize
    for col in prep["numeric_cols"]:
        values = pd.to_numeric(X[col], errors="coerce")
        values = values.fillna(prep["numeric_medians"][col]).to_numpy(dtype=float)
        values = (values - prep["numeric_means"][col]) / prep["numeric_stds"][col]
        blocks.append(values.reshape(-1, 1))

    # Categorical: impute + one-hot encode
    for col in prep["categorical_cols"]:
        values = X[col].astype("object").fillna(
            prep["category_modes"][col]
        ).astype(str).to_numpy()

        categories = prep["category_values"][col]
        encoded = np.zeros((len(X), len(categories)), dtype=float)

        for j, category in enumerate(categories):
            encoded[:, j] = (values == category).astype(float)

        blocks.append(encoded)

    if not blocks:
        return np.empty((len(X), 0))

    return np.hstack(blocks)


In [15]:

# ============================================================
# 13. Apply manual preprocessing using the SAME split
# ============================================================

manual_reg_prep = manual_fit_preprocessor(X_reg_train_raw)
manual_clf_prep = manual_fit_preprocessor(X_clf_train_raw)

manual_X_reg_train = manual_transform(X_reg_train_raw, manual_reg_prep)
manual_X_reg_test = manual_transform(X_reg_test_raw, manual_reg_prep)

manual_X_clf_train = manual_transform(X_clf_train_raw, manual_clf_prep)
manual_X_clf_test = manual_transform(X_clf_test_raw, manual_clf_prep)

print("Manual regression shape:", manual_X_reg_train.shape)
print("Manual classification shape:", manual_X_clf_train.shape)


Manual regression shape: (957, 83)
Manual classification shape: (957, 83)


## 14. Manual Linear Regression

The closed-form solution is:

**β = (XᵀX + λI)⁻¹Xᵀy**

A small ridge term is used only for numerical stability. The intercept is represented by a column of ones.

In [16]:

def add_intercept(X):
    return np.column_stack([np.ones(X.shape[0]), X])


def fit_linear_regression_closed_form(X, y, ridge=1e-10):
    Xb = add_intercept(X)

    identity = np.eye(Xb.shape[1])
    identity[0, 0] = 0.0

    A = Xb.T @ Xb + ridge * identity
    b = Xb.T @ y

    try:
        theta = np.linalg.solve(A, b)
    except np.linalg.LinAlgError:
        theta = np.linalg.pinv(A) @ b

    return theta


def predict_linear_regression(X, theta):
    Xb = add_intercept(X)
    return Xb @ theta


start = time.perf_counter()
manual_reg_theta = fit_linear_regression_closed_form(
    manual_X_reg_train,
    y_reg_train
)
manual_reg_train_time = time.perf_counter() - start

start = time.perf_counter()
manual_reg_pred = predict_linear_regression(
    manual_X_reg_test,
    manual_reg_theta
)
manual_reg_pred_time = time.perf_counter() - start

print("Training time :", manual_reg_train_time)
print("Prediction time:", manual_reg_pred_time)


Training time : 0.002649900008691475
Prediction time: 0.00041000000783242285


In [17]:

# ============================================================
# 15. Manual regression metrics
# ============================================================

def manual_mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))


def manual_rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


def manual_r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1.0 - ss_res / ss_tot


manual_reg_mae = manual_mae(y_reg_test, manual_reg_pred)
manual_reg_rmse = manual_rmse(y_reg_test, manual_reg_pred)
manual_reg_r2 = manual_r2(y_reg_test, manual_reg_pred)

print(f"MAE : {manual_reg_mae:.6f}")
print(f"RMSE: {manual_reg_rmse:.6f}")
print(f"R2  : {manual_reg_r2:.6f}")


MAE : 0.112096
RMSE: 0.150645
R2  : 0.145317


## 16. Manual Logistic Regression

The implementation includes:
- sigmoid function
- probability prediction
- thresholding at 0.5
- vectorized gradient descent
- parameter optimization

In [18]:

def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))


def fit_logistic_regression(
    X,
    y,
    learning_rate=0.05,
    epochs=3000,
    tolerance=1e-8,
    l2=0.0
):
    Xb = add_intercept(X)
    y = y.astype(float)

    theta = np.zeros(Xb.shape[1], dtype=float)
    previous_loss = np.inf

    for epoch in range(epochs):
        scores = Xb @ theta
        probabilities = sigmoid(scores)

        error = probabilities - y

        gradient = (Xb.T @ error) / len(y)

        # Do not regularize intercept
        if l2 > 0:
            regularization = (l2 / len(y)) * theta
            regularization[0] = 0.0
            gradient += regularization

        theta -= learning_rate * gradient

        if epoch % 100 == 0 or epoch == epochs - 1:
            eps = 1e-12
            clipped = np.clip(probabilities, eps, 1 - eps)

            loss = -np.mean(
                y * np.log(clipped) +
                (1 - y) * np.log(1 - clipped)
            )

            if l2 > 0:
                loss += (l2 / (2 * len(y))) * np.sum(theta[1:] ** 2)

            if abs(previous_loss - loss) < tolerance:
                break

            previous_loss = loss

    return theta, epoch + 1


def predict_logistic_proba(X, theta):
    return sigmoid(add_intercept(X) @ theta)


def predict_logistic(X, theta, threshold=0.5):
    probabilities = predict_logistic_proba(X, theta)
    return (probabilities >= threshold).astype(int)


In [19]:

# ============================================================
# 17. Train manual Logistic Regression
# ============================================================

start = time.perf_counter()

manual_clf_theta, manual_clf_epochs = fit_logistic_regression(
    manual_X_clf_train,
    y_clf_train,
    learning_rate=0.05,
    epochs=3000,
    tolerance=1e-8,
    l2=0.0
)

manual_clf_train_time = time.perf_counter() - start

start = time.perf_counter()
manual_clf_pred = predict_logistic(
    manual_X_clf_test,
    manual_clf_theta
)
manual_clf_pred_time = time.perf_counter() - start

print("Training time :", manual_clf_train_time)
print("Prediction time:", manual_clf_pred_time)
print("Epochs used:", manual_clf_epochs)


Training time : 0.18287689999851864
Prediction time: 0.00019749999046325684
Epochs used: 3000


In [20]:

# ============================================================
# 18. Manual classification metrics
# ============================================================

def manual_confusion_counts(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    return tp, tn, fp, fn


def manual_accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)


def manual_precision(y_true, y_pred):
    tp, tn, fp, fn = manual_confusion_counts(y_true, y_pred)
    return tp / (tp + fp) if (tp + fp) else 0.0


def manual_recall(y_true, y_pred):
    tp, tn, fp, fn = manual_confusion_counts(y_true, y_pred)
    return tp / (tp + fn) if (tp + fn) else 0.0


def manual_f1(y_true, y_pred):
    p = manual_precision(y_true, y_pred)
    r = manual_recall(y_true, y_pred)
    return 2 * p * r / (p + r) if (p + r) else 0.0


manual_clf_accuracy = manual_accuracy(y_clf_test, manual_clf_pred)
manual_clf_precision = manual_precision(y_clf_test, manual_clf_pred)
manual_clf_recall = manual_recall(y_clf_test, manual_clf_pred)
manual_clf_f1 = manual_f1(y_clf_test, manual_clf_pred)

print(f"Accuracy : {manual_clf_accuracy:.6f}")
print(f"Precision: {manual_clf_precision:.6f}")
print(f"Recall   : {manual_clf_recall:.6f}")
print(f"F1-score : {manual_clf_f1:.6f}")


Accuracy : 0.762500
Precision: 0.772727
Recall   : 0.960452
F1-score : 0.856423


# Part C — Comparison

In [21]:

# ============================================================
# 19. Comparison table
# ============================================================

comparison = pd.DataFrame([
    {
        "Task": "Regression",
        "Implementation": "Scikit-learn",
        "MAE": sk_reg_mae,
        "RMSE": sk_reg_rmse,
        "R2": sk_reg_r2,
        "Accuracy": np.nan,
        "Precision": np.nan,
        "Recall": np.nan,
        "F1": np.nan,
        "Train Time (s)": sk_reg_train_time,
        "Prediction Time (s)": sk_reg_pred_time
    },
    {
        "Task": "Regression",
        "Implementation": "From Scratch",
        "MAE": manual_reg_mae,
        "RMSE": manual_reg_rmse,
        "R2": manual_reg_r2,
        "Accuracy": np.nan,
        "Precision": np.nan,
        "Recall": np.nan,
        "F1": np.nan,
        "Train Time (s)": manual_reg_train_time,
        "Prediction Time (s)": manual_reg_pred_time
    },
    {
        "Task": "Classification",
        "Implementation": "Scikit-learn",
        "MAE": np.nan,
        "RMSE": np.nan,
        "R2": np.nan,
        "Accuracy": sk_clf_accuracy,
        "Precision": sk_clf_precision,
        "Recall": sk_clf_recall,
        "F1": sk_clf_f1,
        "Train Time (s)": sk_clf_train_time,
        "Prediction Time (s)": sk_clf_pred_time
    },
    {
        "Task": "Classification",
        "Implementation": "From Scratch",
        "MAE": np.nan,
        "RMSE": np.nan,
        "R2": np.nan,
        "Accuracy": manual_clf_accuracy,
        "Precision": manual_clf_precision,
        "Recall": manual_clf_recall,
        "F1": manual_clf_f1,
        "Train Time (s)": manual_clf_train_time,
        "Prediction Time (s)": manual_clf_pred_time
    }
])

display(comparison.round(6))


,Task,Implementation,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Train Time (s),Prediction Time (s)
0,Regression,Scikit-learn,0.112096,0.150645,0.145316,NaN,NaN,NaN,NaN,0.012818,0.001032
1,Regression,From Scratch,0.112096,0.150645,0.145317,NaN,NaN,NaN,NaN,0.002650,0.000410
2,Classification,Scikit-learn,NaN,NaN,NaN,0.7500,0.774648,0.932203,0.846154,4.920923,0.000561
3,Classification,From Scratch,NaN,NaN,NaN,0.7625,0.772727,0.960452,0.856423,0.182877,0.000197


## 20. Optimization of the Manual Logistic Regression

The assignment allows optimization through learning-rate tuning, convergence tuning, regularization, feature selection, or improved preprocessing.

The following small manual grid tests different learning rates and L2 regularization strengths using NumPy only. The test set is not used for tuning; training data is used for the optimization selection.

In [22]:

# ============================================================
# 21. Manual hyperparameter tuning
# ============================================================

optimization_results = []

learning_rates = [0.01, 0.03, 0.05, 0.1]
l2_values = [0.0, 0.001, 0.01, 0.1]

for lr in learning_rates:
    for l2 in l2_values:
        start = time.perf_counter()

        theta, epochs_used = fit_logistic_regression(
            manual_X_clf_train,
            y_clf_train,
            learning_rate=lr,
            epochs=3000,
            tolerance=1e-8,
            l2=l2
        )

        elapsed = time.perf_counter() - start

        train_pred = predict_logistic(
            manual_X_clf_train,
            theta
        )

        train_f1 = manual_f1(y_clf_train, train_pred)
        train_accuracy = manual_accuracy(y_clf_train, train_pred)

        optimization_results.append({
            "learning_rate": lr,
            "l2": l2,
            "epochs": epochs_used,
            "train_accuracy": train_accuracy,
            "train_f1": train_f1,
            "train_time": elapsed
        })

optimization_df = pd.DataFrame(optimization_results)

# Select using training F1, then training accuracy.
optimization_df = optimization_df.sort_values(
    ["train_f1", "train_accuracy"],
    ascending=False
).reset_index(drop=True)

display(optimization_df)


,learning_rate,l2,epochs,train_accuracy,train_f1,train_time
0,0.03,0.000,3000,0.762800,0.853264,0.292928
1,0.03,0.001,3000,0.762800,0.853264,0.284240
2,0.03,0.010,3000,0.762800,0.853264,0.319526
3,0.05,0.100,3000,0.763845,0.853056,0.209807
4,0.03,0.100,3000,0.761755,0.852713,0.213755
5,0.05,0.000,3000,0.761755,0.851562,0.198538
6,0.05,0.001,3000,0.761755,0.851562,0.281867
7,0.05,0.010,3000,0.761755,0.851562,0.180590
8,0.10,0.000,3000,0.764890,0.851485,0.173977
9,0.10,0.001,3000,0.764890,0.851485,0.223996


In [23]:

# ============================================================
# 22. Evaluate selected optimized configuration on test set
# ============================================================

best_config = optimization_df.iloc[0]

optimized_lr = float(best_config["learning_rate"])
optimized_l2 = float(best_config["l2"])

start = time.perf_counter()

optimized_theta, optimized_epochs = fit_logistic_regression(
    manual_X_clf_train,
    y_clf_train,
    learning_rate=optimized_lr,
    epochs=5000,
    tolerance=1e-8,
    l2=optimized_l2
)

optimized_train_time = time.perf_counter() - start

start = time.perf_counter()
optimized_pred = predict_logistic(
    manual_X_clf_test,
    optimized_theta
)
optimized_pred_time = time.perf_counter() - start

optimized_accuracy = manual_accuracy(y_clf_test, optimized_pred)
optimized_precision = manual_precision(y_clf_test, optimized_pred)
optimized_recall = manual_recall(y_clf_test, optimized_pred)
optimized_f1 = manual_f1(y_clf_test, optimized_pred)

print("Selected learning rate:", optimized_lr)
print("Selected L2:", optimized_l2)
print("Epochs:", optimized_epochs)
print(f"Accuracy : {optimized_accuracy:.6f}")
print(f"Precision: {optimized_precision:.6f}")
print(f"Recall   : {optimized_recall:.6f}")
print(f"F1-score : {optimized_f1:.6f}")
print(f"Train time: {optimized_train_time:.6f} s")
print(f"Prediction time: {optimized_pred_time:.6f} s")


Selected learning rate: 0.03
Selected L2: 0.0
Epochs: 5000
Accuracy : 0.762500
Precision: 0.772727
Recall   : 0.960452
F1-score : 0.856423
Train time: 0.313527 s
Prediction time: 0.000182 s


## 23. Final Comparison Including Optimized Manual Logistic Regression

In [24]:

final_comparison = pd.DataFrame([
    {
        "Task": "Regression",
        "Implementation": "Scikit-learn",
        "MAE": sk_reg_mae,
        "RMSE": sk_reg_rmse,
        "R2": sk_reg_r2,
        "Accuracy": np.nan,
        "Precision": np.nan,
        "Recall": np.nan,
        "F1": np.nan,
        "Train Time (s)": sk_reg_train_time,
        "Prediction Time (s)": sk_reg_pred_time
    },
    {
        "Task": "Regression",
        "Implementation": "From Scratch",
        "MAE": manual_reg_mae,
        "RMSE": manual_reg_rmse,
        "R2": manual_reg_r2,
        "Accuracy": np.nan,
        "Precision": np.nan,
        "Recall": np.nan,
        "F1": np.nan,
        "Train Time (s)": manual_reg_train_time,
        "Prediction Time (s)": manual_reg_pred_time
    },
    {
        "Task": "Classification",
        "Implementation": "Scikit-learn",
        "MAE": np.nan,
        "RMSE": np.nan,
        "R2": np.nan,
        "Accuracy": sk_clf_accuracy,
        "Precision": sk_clf_precision,
        "Recall": sk_clf_recall,
        "F1": sk_clf_f1,
        "Train Time (s)": sk_clf_train_time,
        "Prediction Time (s)": sk_clf_pred_time
    },
    {
        "Task": "Classification",
        "Implementation": "From Scratch",
        "MAE": np.nan,
        "RMSE": np.nan,
        "R2": np.nan,
        "Accuracy": manual_clf_accuracy,
        "Precision": manual_clf_precision,
        "Recall": manual_clf_recall,
        "F1": manual_clf_f1,
        "Train Time (s)": manual_clf_train_time,
        "Prediction Time (s)": manual_clf_pred_time
    },
    {
        "Task": "Classification",
        "Implementation": "Optimized From Scratch",
        "MAE": np.nan,
        "RMSE": np.nan,
        "R2": np.nan,
        "Accuracy": optimized_accuracy,
        "Precision": optimized_precision,
        "Recall": optimized_recall,
        "F1": optimized_f1,
        "Train Time (s)": optimized_train_time,
        "Prediction Time (s)": optimized_pred_time
    }
])

display(final_comparison.round(6))


,Task,Implementation,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Train Time (s),Prediction Time (s)
0,Regression,Scikit-learn,0.112096,0.150645,0.145316,NaN,NaN,NaN,NaN,0.012818,0.001032
1,Regression,From Scratch,0.112096,0.150645,0.145317,NaN,NaN,NaN,NaN,0.002650,0.000410
2,Classification,Scikit-learn,NaN,NaN,NaN,0.7500,0.774648,0.932203,0.846154,4.920923,0.000561
3,Classification,From Scratch,NaN,NaN,NaN,0.7625,0.772727,0.960452,0.856423,0.182877,0.000197
4,Classification,Optimized From Scratch,NaN,NaN,NaN,0.7625,0.772727,0.960452,0.856423,0.313527,0.000182


# 24. Additional Diagnostics

The following cells help explain the differences between implementations without changing the required evaluation protocol.

In [25]:

# Confusion matrix for the final manual model
tp, tn, fp, fn = manual_confusion_counts(y_clf_test, optimized_pred)

confusion = pd.DataFrame(
    [[tn, fp], [fn, tp]],
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"]
)

display(confusion)


,Predicted 0,Predicted 1
Actual 0,13,50
Actual 1,7,170


In [26]:

# Check that classification does not contain actual_productivity
classification_feature_columns = X_clf_raw.columns.tolist()

print("actual_productivity" in classification_feature_columns)
print("Classification features:")
print(classification_feature_columns)


False
Classification features:
['date', 'quarter', 'department', 'day', 'team', 'targeted_productivity', 'smv', 'wip', 'over_time', 'incentive', 'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers']


# 25. Key Observations

Run the notebook first, then use the generated numbers to complete the observations below.

1. **Regression performance:** Compare MAE, RMSE and R² between Scikit-learn and the manual Linear Regression implementation.
2. **Classification performance:** Compare accuracy, precision, recall and F1-score between the two Logistic Regression implementations.
3. **Runtime:** Compare training and prediction time. Runtime can differ because Scikit-learn uses highly optimized numerical routines and solver implementations.
4. **Manual optimization:** Record how learning rate, L2 regularization and convergence settings affect the manual Logistic Regression.
5. **Fairness:** Both implementations use the same train-test samples and target definitions.
6. **Classification leakage prevention:** `actual_productivity` is excluded from classification inputs because it is used to construct `MeetsTarget`.
7. **Final explanation:** State whether the optimized manual implementation reduced the performance/runtime gap and explain the observed differences using the actual results generated above.

In [27]:

# ============================================================
# 26. Automatically generate a compact results summary
# ============================================================

print("=" * 70)
print("FINAL RESULTS SUMMARY")
print("=" * 70)

print("\nRegression")
print(f"Scikit-learn -> MAE={sk_reg_mae:.4f}, RMSE={sk_reg_rmse:.4f}, R2={sk_reg_r2:.4f}")
print(f"From Scratch -> MAE={manual_reg_mae:.4f}, RMSE={manual_reg_rmse:.4f}, R2={manual_reg_r2:.4f}")

print("\nClassification")
print(
    f"Scikit-learn -> Accuracy={sk_clf_accuracy:.4f}, "
    f"Precision={sk_clf_precision:.4f}, Recall={sk_clf_recall:.4f}, F1={sk_clf_f1:.4f}"
)
print(
    f"From Scratch -> Accuracy={manual_clf_accuracy:.4f}, "
    f"Precision={manual_clf_precision:.4f}, Recall={manual_clf_recall:.4f}, F1={manual_clf_f1:.4f}"
)
print(
    f"Optimized From Scratch -> Accuracy={optimized_accuracy:.4f}, "
    f"Precision={optimized_precision:.4f}, Recall={optimized_recall:.4f}, F1={optimized_f1:.4f}"
)

print("\nNote: Timing values depend on the machine, Python environment and BLAS implementation.")


FINAL RESULTS SUMMARY

Regression
Scikit-learn -> MAE=0.1121, RMSE=0.1506, R2=0.1453
From Scratch -> MAE=0.1121, RMSE=0.1506, R2=0.1453

Classification
Scikit-learn -> Accuracy=0.7500, Precision=0.7746, Recall=0.9322, F1=0.8462
From Scratch -> Accuracy=0.7625, Precision=0.7727, Recall=0.9605, F1=0.8564
Optimized From Scratch -> Accuracy=0.7625, Precision=0.7727, Recall=0.9605, F1=0.8564

Note: Timing values depend on the machine, Python environment and BLAS implementation.


## 27. Reproducibility Checklist

- [x] Fixed random seed
- [x] One fixed train-test split reused for comparisons
- [x] Regression target = `actual_productivity`
- [x] Classification target = `MeetsTarget`
- [x] `actual_productivity` excluded from classification features
- [x] Missing values handled
- [x] Categorical features encoded
- [x] Numeric features scaled
- [x] Scikit-learn Linear Regression
- [x] Scikit-learn Logistic Regression
- [x] Manual Linear Regression
- [x] Manual Logistic Regression
- [x] Manual prediction functions
- [x] Manual evaluation metrics
- [x] Training and prediction timing
- [x] Manual optimization
- [x] Comparison tables

**Before submission:** place the notebook and dataset in your project as appropriate, run all cells from top to bottom, verify that all outputs are present, and add a README containing your actual observations and GitHub repository information.